# 01b — Capa tipada de variables núcleo

Lee el Parquet **ya armonizado del notebook 01** (`data/01_armonizado/{hogar,individual}/`)
-- no vuelve a tocar `data/eph_cache/` ni re-parsea DBF. Selecciona las variables núcleo
de `src/tipado_nucleo.ESQUEMA_NUCLEO` (ingresos, empleo, vivienda, estrategias del
hogar), las tipa con el **mismo dtype en todos los trimestres**, reporta cualquier valor
que no haya podido parsearse (nunca lo imputa ni lo descarta en silencio), y escribe a
`data/02_nucleo/`.

**Parámetros** (celda de setup): `TRIMESTRES` (subconjunto opcional) y `REHACER`
(`False` por defecto -- con `REHACER=False`, una partición se reprocesa si no existe
**o si el Parquet de origen en `01_armonizado/` es más nuevo** que el de `02_nucleo/`,
no solo si falta). Corrida parcial (`TRIMESTRES` acotado): `p21_series_trimestral.csv`/
`formato_decimal_monetario.csv` se escriben con sufijo `_parcial`, no pisan los globales.

El esquema está validado contra el codebook oficial de INDEC
(`data/EPH_tot_urbano_estructura_bases_2025.pdf`) donde lo cubre -- ver los docstrings
de `tipado_nucleo.py` y el plan de esta tarea para el detalle de qué está confirmado y
qué sigue inferido.

In [ ]:
import sys
import time
from pathlib import Path

import pandas as pd


def _raiz_repo(desde: Path | None = None) -> Path:
    actual = (desde or Path.cwd()).resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pytest.ini").exists():
            return candidato
    raise FileNotFoundError(f"no se encontró la raíz del repo subiendo desde {actual}")


REPO = _raiz_repo()
sys.path.insert(0, str(REPO / "src"))

from armonizacion import trimestres_publicados, es_historico
import tipado_nucleo as tn
import pipeline_meta as pm

TRIMESTRES: list[tuple[int, int]] | None = None  # subconjunto opcional; None = todos
REHACER: bool = False  # True fuerza reprocesar aunque la partición esté al día

ARMONIZADO = REPO / "data" / "01_armonizado"
NUCLEO = REPO / "data" / "02_nucleo"
META = REPO / "data" / "meta"
NUCLEO.mkdir(parents=True, exist_ok=True)
META.mkdir(parents=True, exist_ok=True)

trimestres = TRIMESTRES if TRIMESTRES is not None else trimestres_publicados()
ES_CORRIDA_PARCIAL = TRIMESTRES is not None
COMMIT_GIT = pm.obtener_commit_git(REPO)
print(f"{len(trimestres)} trimestres a procesar" +
      (" -- CORRIDA PARCIAL (TRIMESTRES acotado)" if ES_CORRIDA_PARCIAL
       else "; se procesan los que ya tengan Parquet del notebook 01"))

## 1. Tipado por trimestre + reporte de no-parseables

In [ ]:
no_parseables = []
escritos = []
reusados = []
saltados = []
estado_p21_por_trimestre = []
formato_decimal_registros = []
ESQUEMA_HASH = tn.hash_esquema()

# Hash de esquema con el que se tipó cada partición ya escrita -- para las que
# se reusan (no se re-tipan), este es el dato correcto a guardar en el resumen,
# no ESQUEMA_HASH actual (que podría ser más nuevo si el esquema cambió y esta
# corrida no forzó REHACER). Se lee una sola vez antes del loop, no por partición.
_resumen_path = META / "resumen_02_nucleo.csv"
if _resumen_path.exists():
    _resumen_previo = pd.read_csv(_resumen_path)
    HASH_PREVIO_POR_PARTICION = {
        (f["anio"], f["trimestre"], f["tipo"]): f.get("esquema_hash")
        for f in _resumen_previo.to_dict("records")
    }
else:
    HASH_PREVIO_POR_PARTICION = {}

t0 = time.time()
for anio, trimestre in trimestres:
    historico = es_historico(anio, trimestre)
    hogar_origen = pm.ruta_particion(ARMONIZADO, "hogar", anio, trimestre)
    ind_origen = pm.ruta_particion(ARMONIZADO, "individual", anio, trimestre)
    if not hogar_origen.exists() or not ind_origen.exists():
        saltados.append((anio, trimestre, "sin Parquet del notebook 01 (pendiente_rar)"))
        continue

    hogar = pd.read_parquet(hogar_origen)
    individual = pd.read_parquet(ind_origen)

    filas_resumen_trimestre = []
    for tipo, df, origen_path in (("hogar", hogar, hogar_origen), ("individual", individual, ind_origen)):
        destino_path = pm.ruta_particion(NUCLEO, tipo, anio, trimestre)
        desactualizada = pm.particion_desactualizada(origen_path, destino_path)

        # Chequeo de formato decimal: barato (no requiere tipar), se corre siempre
        # -- incluso cuando la partición se reusa -- para no perder cobertura en
        # formato_decimal_monetario.csv en corridas donde todo se reusa (antes,
        # el chequeo vivía solo en la rama de re-tipado: una corrida 100% reusada
        # pisaba el CSV con 0 filas, perdiendo el diagnóstico ya calculado en una
        # corrida anterior).
        for variable in tn.ESQUEMA_NUCLEO:
            if tn.ESQUEMA_NUCLEO[variable]["base"] != tipo or variable not in df.columns:
                continue
            if tn.ESQUEMA_NUCLEO[variable]["regla_parseo"] == "monetario":
                formato = tn.detectar_formato_decimal(df[variable])
                formato_decimal_registros.append({"anio": anio, "trimestre": trimestre, "tipo": tipo, "variable": variable, **formato})

        if not desactualizada and not REHACER:
            # 01_armonizado no cambió desde la última vez que se tipó -- se
            # relee lo ya escrito (liviano) en vez de re-tipar todo de nuevo.
            resultado = pd.read_parquet(destino_path)
            hash_previo = HASH_PREVIO_POR_PARTICION.get((anio, trimestre, tipo))
            if pd.isna(hash_previo) if hash_previo is not None else True:
                # No hay hash registrado para esta partición (falta la clave o
                # es NaN) -- se guarda "desconocido" en vez de asumir que
                # coincide con el esquema vigente (ESQUEMA_HASH). La notebook
                # 02 trata cualquier valor distinto al hash vigente como
                # desactualizado, así que "desconocido" cae ahí solo.
                hash_previo = "desconocido"
            reusados.append((anio, trimestre, tipo, len(resultado)))
            filas_resumen_trimestre.append({
                "anio": anio, "trimestre": trimestre, "tipo": tipo,
                "filas_escritas": len(resultado), "saltado": True, "no_parseables": False,
                "esquema_hash": hash_previo,
            })
            continue

        claves = [c for c in ("CODUSU", "NRO_HOGAR", "COMPONENTE", "ANO4", "TRIMESTRE", "AGLOMERADO") if c in df.columns]
        variables_presentes = [v for v in tn.ESQUEMA_NUCLEO if tn.ESQUEMA_NUCLEO[v]["base"] == tipo and v in df.columns]
        resultado = df[claves].copy()
        hubo_no_parseables = False
        for variable in variables_presentes:
            tipada = tn.tipar_variable(df, variable)
            reporte = tn.reporte_no_parseables(df[variable], tipada, anio, trimestre, tipo, variable)
            hubo_no_parseables = hubo_no_parseables or bool(reporte)
            no_parseables.extend(reporte)
            resultado[variable] = tipada
        if tipo == "hogar":
            resultado["ingreso_no_declarado"] = tn.ingreso_no_declarado(df, historico)

        destino_path.parent.mkdir(parents=True, exist_ok=True)
        resultado.to_parquet(destino_path, index=False)
        escritos.append((anio, trimestre, tipo, len(resultado)))
        filas_resumen_trimestre.append({
            "anio": anio, "trimestre": trimestre, "tipo": tipo,
            "filas_escritas": len(resultado), "saltado": False, "no_parseables": hubo_no_parseables,
            "esquema_hash": ESQUEMA_HASH,
        })

    pm.actualizar_resumen_trimestre(META / "resumen_02_nucleo.csv", filas_resumen_trimestre)

    # datos para la sección 3 (P21=0 / P21=-9, general y asalariados) -- se arman
    # acá mismo pase, para no releer individual de nuevo.
    estado = pd.to_numeric(individual["ESTADO"], errors="coerce")
    cat_ocup = pd.to_numeric(individual["CAT_OCUP"], errors="coerce")
    p21 = tn.parsear_monetario(individual["P21"])

    ocupados = p21[estado == 1]
    validos = ocupados[ocupados != -9] if not historico else ocupados
    n_total = len(validos)
    n_cero = (validos == 0).sum()
    pct_p21_cero = (n_cero / n_total) if n_total else None

    if not historico:
        n_ocupados = len(ocupados)
        n_m9 = (ocupados == -9).sum()
        pct_p21_m9 = (n_m9 / n_ocupados) if n_ocupados else None
    else:
        pct_p21_m9 = None

    asalariados = p21[(estado == 1) & (cat_ocup == 3)]
    validos_asal = asalariados[asalariados != -9] if not historico else asalariados
    n_total_asal = len(validos_asal)
    n_cero_asal = (validos_asal == 0).sum()
    pct_p21_cero_asalariados = (n_cero_asal / n_total_asal) if n_total_asal else None

    estado_p21_por_trimestre.append({
        "anio": anio, "trimestre": trimestre, "periodo": pd.Period(f"{anio}Q{trimestre}", freq="Q"),
        "pct_p21_cero": pct_p21_cero,
        "pct_p21_menos9": pct_p21_m9,
        "pct_p21_cero_asalariados": pct_p21_cero_asalariados,
    })

duracion_total = time.time() - t0
print(f"tiempo total: {duracion_total:.1f}s")
print(f"escritos: {len(escritos)} -- reusados (al día): {len(reusados)} -- saltados: {len(saltados)}")
for s in saltados:
    print("  saltado:", s)

In [ ]:
reporte_no_parseables_df = pd.DataFrame(no_parseables)
if reporte_no_parseables_df.empty:
    print("reporte de no-parseables: VACÍO -- ningún valor no-nulo falló al tipar en ninguna variable/trimestre.")
else:
    print(f"reporte de no-parseables: {len(reporte_no_parseables_df)} filas (variable/trimestre/valor con conteo)")
    display(reporte_no_parseables_df.sort_values("cantidad", ascending=False))

### 1.1 Verificación de formato decimal único por trimestre

Para cada variable monetaria y cada trimestre, `detectar_formato_decimal` chequea si la columna mezcla coma y punto decimal en el mismo trimestre (la anomalía real sería esa mezcla puertas adentro de un trimestre, no que distintos trimestres usen convenciones distintas -- eso último ya está cubierto por `parsear_monetario`).

In [ ]:
COLUMNAS_FORMATO_DECIMAL = ["anio", "trimestre", "tipo", "variable", "tiene_coma", "tiene_punto", "mezcla"]
formato_decimal_df = pd.DataFrame(formato_decimal_registros, columns=COLUMNAS_FORMATO_DECIMAL)
mezclas = formato_decimal_df[formato_decimal_df["mezcla"] == True]
print(f"{len(formato_decimal_df)} combinaciones variable-monetaria x trimestre revisadas"
      + (" (0 -- no hay trimestres con Parquet de origen en esta corrida)" if formato_decimal_df.empty else ""))
print(f"trimestres con mezcla de coma Y punto en la misma variable: {len(mezclas)}")
if len(mezclas):
    display(mezclas)
elif not formato_decimal_df.empty:
    print("Ninguno -- cada trimestre usa un solo separador decimal por variable (aunque distinto entre trimestres, nunca mezclado dentro de uno).")

usa_punto = formato_decimal_df[formato_decimal_df["tiene_punto"] == True]
print(f"\ncombinaciones variable x trimestre que usan punto decimal: {len(usa_punto)}")
usa_punto[["anio", "trimestre", "tipo", "variable"]].drop_duplicates().sort_values(["anio", "trimestre"])

In [ ]:
sufijo = "_parcial" if ES_CORRIDA_PARCIAL else ""
destino_formato = META / f"formato_decimal_monetario{sufijo}.csv"
formato_decimal_df.to_csv(destino_formato, index=False)
print(f"guardado: {destino_formato} ({len(formato_decimal_df)} filas)" +
      (" -- CORRIDA PARCIAL, no pisa el global" if ES_CORRIDA_PARCIAL else ""))

## 2. Flag `ingreso_no_declarado` -- casos de `DECCFR` nulo capturados

Documentado explícitamente (pedido de la tarea): hogares donde `DECCFR` es nulo (no se
calculó el decil) pero `PONDIH=0` -- el flag los captura igual que a los `DECCFR==12`,
aunque no comparten el código.

In [ ]:
casos_deccfr_nulo = []
for anio, trimestre in trimestres:
    if es_historico(anio, trimestre):
        continue
    p = ARMONIZADO / "hogar" / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet"
    if not p.exists():
        continue
    columnas_disponibles = __import__("pyarrow.parquet", fromlist=["read_schema"]).read_schema(p).names
    if "DECCFR" not in columnas_disponibles or "PONDIH" not in columnas_disponibles:
        continue
    df = pd.read_parquet(p, columns=["CODUSU", "NRO_HOGAR", "DECCFR", "PONDIH"])
    deccfr = tn.parsear_entero_con_blancos(df["DECCFR"])
    pondih = pd.to_numeric(df["PONDIH"], errors="coerce")
    nulo_con_pondih0 = deccfr.isna() & (pondih == 0)
    if nulo_con_pondih0.any():
        for _, fila in df[nulo_con_pondih0].iterrows():
            casos_deccfr_nulo.append({"anio": anio, "trimestre": trimestre, "CODUSU": fila["CODUSU"], "NRO_HOGAR": fila["NRO_HOGAR"]})

print(f"{len(casos_deccfr_nulo)} hogares con DECCFR nulo y PONDIH=0 (capturados por ingreso_no_declarado igual):")
pd.DataFrame(casos_deccfr_nulo)

## 3. P21: no-respuesta (-9) e ingreso cero, ocupados y asalariados (patrón mostrado, sin concluir)

In [ ]:
import matplotlib.pyplot as plt

serie_p21 = pd.DataFrame(estado_p21_por_trimestre).sort_values(["anio", "trimestre"])
serie_p21["fecha"] = serie_p21["periodo"].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(serie_p21["fecha"], serie_p21["pct_p21_cero"] * 100, marker=".", label="% ocupados con P21=0")
ax.plot(serie_p21["fecha"], serie_p21["pct_p21_menos9"] * 100, marker=".", label="% ocupados con P21=-9 (solo era regular)")
ax.plot(serie_p21["fecha"], serie_p21["pct_p21_cero_asalariados"] * 100, marker=".", label="% asalariados (CAT_OCUP=3) con P21=0")
ax.axvline(pd.Timestamp("2016-01-01"), linestyle="--", color="gray", label="corte histórico/regular (2016)")
ax.set_ylabel("%")
ax.set_title("P21: no-respuesta e ingreso cero, ocupados y asalariados (excluye P21=-9 de la era regular en pct_p21_cero*)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print("--- % ocupados con P21=0 ---")
print("promedio pre-2016:", serie_p21.loc[serie_p21.anio <= 2015, "pct_p21_cero"].mean())
print("promedio post-2016:", serie_p21.loc[serie_p21.anio > 2015, "pct_p21_cero"].mean())

print("\n--- % ocupados con P21=-9 (solo era regular) ---")
print("promedio:", serie_p21["pct_p21_menos9"].mean())

print("\n--- % asalariados (CAT_OCUP=3) con P21=0 ---")
print("promedio pre-2016:", serie_p21.loc[serie_p21.anio <= 2015, "pct_p21_cero_asalariados"].mean())
print("promedio post-2016:", serie_p21.loc[serie_p21.anio > 2015, "pct_p21_cero_asalariados"].mean())

In [ ]:
sufijo = "_parcial" if ES_CORRIDA_PARCIAL else ""
destino_p21 = META / f"p21_series_trimestral{sufijo}.csv"
serie_p21.drop(columns=["periodo", "fecha"]).to_csv(destino_p21, index=False)
print(f"guardado: {destino_p21} ({len(serie_p21)} filas)" +
      (" -- CORRIDA PARCIAL, no pisa el global" if ES_CORRIDA_PARCIAL else ""))

**Patrón, mostrado sin concluir** (no se elige entre las hipótesis de D3):

- `% ocupados con P21=-9` (solo existe en la era regular, ver diagnóstico §6): se
  reporta el promedio de toda la serie 2016-2025, sin comparar contra "antes"
  porque no hay "antes" para este código.
- `% asalariados con P21=0` sigue de cerca a la serie general de ocupados (misma
  forma, niveles algo distintos) -- no aparece un quiebre propio de los
  asalariados que no esté ya en la serie general.
- Ninguna de las series nuevas muestra un escalón adicional justo en 2016 que no
  estuviera ya en la serie original de ocupados.

## 4. Validación DuckDB — ninguna columna núcleo numérica debe quedar VARCHAR

In [ ]:
import duckdb

con = duckdb.connect()
for tipo in ("hogar", "individual"):
    glob = str(NUCLEO / tipo / "**" / "*.parquet")
    con.sql(f"CREATE OR REPLACE VIEW nucleo_{tipo} AS SELECT * FROM read_parquet('{glob}', union_by_name=true)")

for tipo in ("hogar", "individual"):
    tipos_columnas = con.sql(f"DESCRIBE SELECT * FROM nucleo_{tipo}").df()
    varchar_inesperado = tipos_columnas[
        (tipos_columnas["column_type"] == "VARCHAR") & (~tipos_columnas["column_name"].isin(["CODUSU"]))
    ]
    print(f"{tipo}: columnas VARCHAR fuera de CODUSU: {len(varchar_inesperado)}")
    if len(varchar_inesperado):
        display(varchar_inesperado)
    assert varchar_inesperado.empty, f"hay columnas núcleo de {tipo} que quedaron VARCHAR"
print("OK: ninguna columna núcleo numérica quedó VARCHAR (CODUSU es la única string esperada).")

## 5. Resumen final

In [ ]:
n_filas_totales = sum(f for *_, f in escritos)
pm.registrar_corrida(
    META / "corridas.csv",
    notebook="01b_capa_tipada_nucleo",
    parametros={"TRIMESTRES": TRIMESTRES, "REHACER": REHACER},
    n_trimestres=len(trimestres),
    n_filas=n_filas_totales,
    duracion_segundos=duracion_total,
    commit_git=COMMIT_GIT,
)

print("Trimestres escritos (nuevo):", len({(a, t) for a, t, *_ in escritos}))
print("Trimestres reusados (al día):", len({(a, t) for a, t, *_ in reusados}))
print("Trimestres saltados:", len(saltados))
print("Filas en reporte de no-parseables:", len(reporte_no_parseables_df))
print("Hogares con DECCFR nulo + PONDIH=0 (documentados en ingreso_no_declarado):", len(casos_deccfr_nulo))